# init notebook setting up the backend. 

Do not edit the notebook, it contains import and helpers for the demo

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F_resources%2F00-setup&demo_name=lakehouse-fsi-smart-claims&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-smart-claims%2F_resources%2F00-setup&version=1">

In [0]:
%run ../config

## Configuration file

Please change your catalog and schema here to run the demo on a different catalog.

<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F_resources%2F00-setup&demo_name=lakehouse-fsi-smart-claims&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-smart-claims%2F_resources%2F00-setup&version=1">

In [0]:
%run ./00-global-setup-v2


# Technical Setup notebook. Hide this cell results
Initialize dataset to the current user and cleanup data when reset_all_data is set to true

Do not edit

In [0]:
print('run done')
DBDemos.setup_schema(catalog, db, False, volume_name)
volume_folder = f"/Volumes/{catalog}/{db}/{volume_name}"

run done
USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_fsi_smart_claims`


In [0]:
import os
import requests
import timeit
import time
import collections
 
if DBDemos.is_any_folder_empty([volume_folder+"/Accidents", volume_folder+"/Claims", volume_folder+"/Policies", volume_folder+"/Images", volume_folder+"/Telematics"]):
  print(f'Downloading raw data under {volume_folder}...')
  #Accidents
  DBDemos.download_file_from_git(volume_folder+'/Accidents/metadata', "databricks-demos", "dbdemos-dataset", "/fsi/smart-claims/Accidents/metadata")
  DBDemos.download_file_from_git(volume_folder+'/Accidents/images', "databricks-demos", "dbdemos-dataset", "/fsi/smart-claims/Accidents/images")
  #Claims
  DBDemos.download_file_from_git(volume_folder+'/Claims', "databricks-demos", "dbdemos-dataset", "/fsi/smart-claims/Claims/Claims")
  #Policies
  DBDemos.download_file_from_git(volume_folder+'/Policies', "databricks-demos", "dbdemos-dataset", "/fsi/smart-claims/Policies")
  #Telematics
  DBDemos.download_file_from_git(volume_folder+'/Telematics', "databricks-demos", "dbdemos-dataset", "/fsi/smart-claims/Telematics")
  #training images
  DBDemos.download_file_from_git(volume_folder+'/Images', "databricks-demos", "dbdemos-dataset", "/fsi/smart-claims/Images")
else:
  print("data already existing.")

saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/metadata/image_metadata.csv
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/1_Low.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/1_High.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/3_Medium.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/3_High.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_Low.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_Medium.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/4_High.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/3_Low.jpg
saving /Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/1_Medium.jpg
saving /Volumes/mai

In [0]:
import numpy as np
import pandas as pd
import pyspark.sql.functions as F

In [0]:
#Force torch to local filestore to properly support serverless workspaces
try:
    import os
    import tempfile
    import torch

    file_store_path = os.path.join(tempfile.gettempdir(), os.environ["VIRTUAL_ENV"].split("/")[-1])
    store = torch.distributed.FileStore(file_store_path, world_size=1)
    torch.distributed.init_process_group(backend="gloo", rank=0, world_size=1, store=store)
except:
    pass